## AI4PH Course, Ch07 GAN, Example 02
## VAE for MNIST (section 6)

**Pytorch translation**, based on the Companion Notebook (in Keras) for Chapter 17, Image Generation in F. Chollet & M. Watson, Deep Learning with Python, (3rd Edition) , with additional functions to explore the latent space

In [ ]:
#!pip install -q torch torchvision torchaudio diffusers transformers accelerate sentencepiece matplotlib pillow datasets

### Mount Google Drive or use local directory

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CHECKPOINT_DIR = '/content/drive/MyDrive/IA4PH_c2526/07_GAN_example_02'
import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print('Checkpoint directory:', CHECKPOINT_DIR)

In [ ]:
CHECKPOINT_DIR = './07_GAN_example_02'
import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print('Checkpoint directory:', CHECKPOINT_DIR)

#### Imports & Device

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchinfo import summary
import numpy as np
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, ConcatDataset

import matplotlib.pyplot as plt
import time
import imageio

print("PyTorch version:", torch.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

### 1. Building a VAE: Encoder, Sampler, Decoder

In [ ]:
latent_dim = 2

class Encoder(nn.Module):
    def __init__(self, latent_dim=2):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1)
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(64 * 7 * 7, 16)
        self.z_mean = nn.Linear(16, latent_dim)
        self.z_log_var = nn.Linear(16, latent_dim)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.flatten(x)
        x = F.relu(self.fc(x))
        z_mean = self.z_mean(x)
        z_log_var = self.z_log_var(x)
        return z_mean, z_log_var

encoder = Encoder(latent_dim).to(device)
encoder

In [ ]:
summary(encoder, input_size=(1, 1, 28, 28), depth=3)

In [ ]:
class Sampler(nn.Module):
    def forward(self, z_mean, z_log_var):
        epsilon = torch.randn_like(z_mean)
        return z_mean + torch.exp(0.5 * z_log_var) * epsilon

sampler = Sampler()

In [ ]:
class Decoder(nn.Module):
    def __init__(self, latent_dim=2):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 7 * 7 * 64)
        self.deconv1 = nn.ConvTranspose2d(
            64, 64, kernel_size=3, stride=2, padding=1, output_padding=1
        )
        self.deconv2 = nn.ConvTranspose2d(
            64, 32, kernel_size=3, stride=2, padding=1, output_padding=1
        )
        self.out = nn.Conv2d(32, 1, kernel_size=3, padding=1)

    def forward(self, z):
        x = F.relu(self.fc(z))
        x = x.view(-1, 64, 7, 7)
        x = F.relu(self.deconv1(x))
        x = F.relu(self.deconv2(x))
        return torch.sigmoid(self.out(x))

decoder = Decoder(latent_dim).to(device)
decoder

In [ ]:
summary(decoder, input_size=(1, latent_dim), depth=3)

In [ ]:
class VAE(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.sampler = Sampler()

    def forward(self, x):
        z_mean, z_log_var = self.encoder(x)
        z = self.sampler(z_mean, z_log_var)
        reconstruction = self.decoder(z)
        return reconstruction, z_mean, z_log_var

    def compute_loss(self, x):
        reconstruction, z_mean, z_log_var = self(x)
        reconstruction_loss = F.binary_cross_entropy(
            reconstruction, x, reduction="none"
        ).sum(dim=(1, 2, 3)).mean()
        kl_loss = -0.5 * (
            1 + z_log_var - z_mean.pow(2) - z_log_var.exp()
        ).sum(dim=1).mean()
        total_loss = reconstruction_loss + kl_loss
        return total_loss, reconstruction_loss, kl_loss

#### Save & Load Checkpoints

In [ ]:
def save_checkpoint(epoch):
    path = f"{CHECKPOINT_DIR}/vae_epoch_{epoch}.pt"
    torch.save({'epoch': epoch,
                'model': vae.state_dict(),
                'opt': optimizer.state_dict()}, path)
    print('Saved checkpoint:', path)

def load_checkpoint(path):
    ckpt = torch.load(path, map_location=device)
    vae.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['opt'])
    print('Loaded:', path)
    return ckpt['epoch']

### 2. Preparing the training data set

In [ ]:
transform = transforms.ToTensor()
train_ds = datasets.MNIST(root=".", train=True, download=True, transform=transform)
test_ds = datasets.MNIST(root=".", train=False, download=True, transform=transform)
mnist_ds = ConcatDataset([train_ds, test_ds])

batch_size = 128
train_loader = DataLoader(mnist_ds, batch_size=batch_size, shuffle=True, drop_last=True)

len(train_loader)

### 3. Training

In [ ]:
def train_vae(model, loader, optimizer, epochs=30):
    model.train()
    history = []
    t0 = time.time()
    t00 = t0
    for epoch in range(epochs):
        total_loss = total_rec = total_kl = 0.0
        for x, _ in loader:
            x = x.to(device)
            optimizer.zero_grad()
            loss, rec, kl = model.compute_loss(x)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            total_rec += rec.item()
            total_kl += kl.item()
            
        history.append(
            {
                "loss": total_loss / len(loader),
                "reconstruction_loss": total_rec / len(loader),
                "kl_loss": total_kl / len(loader),
            }
        )
        
        t1 = time.time()
        print(
            f"Epoch {epoch + 1:02d} | "
            f"loss={history[-1]['loss']:.4f} | "
            f"reconstruction_loss={history[-1]['reconstruction_loss']:.4f} | "
            f"kl_loss={history[-1]['kl_loss']:.4f}"
        )
        print(f"Training time (epoch): {t1 - t00:.1f} s")
        t00 = t1

        # Save model every few epochs
        if (epoch+1) % 5 == 0 or epoch+1 == 1:
            save_checkpoint(epoch)
            print(f"Saved model at epoch {epoch+1}")

    print(f"TOTAL training time: {t1 - t0:.1f} s")
    return history

In [ ]:
vae = VAE(encoder, decoder).to(device)
optimizer = torch.optim.Adam(vae.parameters(), lr=1e-3)
vae_history = train_vae(vae, train_loader, optimizer, epochs=30)

In [ ]:
save_checkpoint(30)

In [ ]:
load_checkpoint('./07_GAN_example_02/vae_epoch_30.pt')

### 4. Exploring the latent space

#### Reconstruction

In [ ]:
def show_reconstructions():
    vae.eval()
    x,_ = next(iter(train_loader))
    x = x[:8].to(device)
    with torch.no_grad():
        rec,_,_ = vae(x)
    fig,ax = plt.subplots(2,8,figsize=(12,4))
    for i in range(8):
        ax[0,i].imshow(x[i].cpu().squeeze(), cmap='gray'); ax[0,i].axis('off')
        ax[1,i].imshow(rec[i].cpu().squeeze(), cmap='gray'); ax[1,i].axis('off')
    plt.show()

In [ ]:
show_reconstructions()

#### Latent Space Visualization

In [ ]:
def plot_latent_space(num_batches=5):
    vae.eval()
    zs=[]; labels=[]
    with torch.no_grad():
        for i,(x,y) in enumerate(train_loader):
            if i>=num_batches: break
            x=x.to(device)
            mu,_ = vae.encoder(x)
            zs.append(mu.cpu()); labels.append(y)
    zs=torch.cat(zs); labels=torch.cat(labels)
    plt.figure(figsize=(12,8))
    plt.scatter(zs[:,0], zs[:,1], c=labels, cmap='tab10', s=5)
    plt.colorbar()
    plt.title('Latent Space (μ)')
    plt.show()

In [ ]:
plot_latent_space(50)

#### Sample a Latent Grid

In [ ]:
def sample_latent_grid(n=20, r=3.0):
    vae.eval()
    xs=np.linspace(-r,r,n)
    ys=np.linspace(-r,r,n)[::-1]
    imgs=[]
    with torch.no_grad():
        for yi in ys:
            row=[]
            for xi in xs:
                z=torch.tensor([[xi,yi]],dtype=torch.float32,device=device)
                img=vae.decoder(z)[0].cpu().squeeze().numpy()
                row.append(img)
            imgs.append(row)
    big=np.block([[imgs[j][i] for i in range(n)] for j in range(n)])
    plt.figure(figsize=(15,15))
    plt.imshow(big,cmap='gray')
    plt.axis('off')
    plt.show()

In [ ]:
sample_latent_grid(30,1.0)

#### GIF: Latent Traversal

In [ ]:
def save_latent_traversal_gif(dim=0, steps=40, r=3.0, fname='traversal.gif'):
    vae.eval()
    vals=np.linspace(-r,r,steps)
    frames=[]
    with torch.no_grad():
        for v in vals:
            z=torch.zeros((1,latent_dim),device=device)
            z[0,dim]=v
            img=vae.decoder(z)[0].cpu().squeeze().numpy()
            img=(img*255).astype(np.uint8)
            frames.append(img)
    path=f"{CHECKPOINT_DIR}/{fname}"
    imageio.mimsave(path, frames, fps=10)
    print('GIF saved:', path)

In [ ]:
save_latent_traversal_gif(dim=0, fname='traversal_dim0.gif')
save_latent_traversal_gif(dim=1, fname='traversal_dim1.gif')